# Milestone 3 — Vizgen MERFISH cross-reference

Mean log2(CPM+1) per **cell type** × **brain area** on the Vizgen Mouse Brain Receptor Map,
with Allen MERFISH label transfer (kNN on overlapping genes), then side-by-side heatmaps and
correlation scatters vs Allen MERFISH (notebook 02). Driven by `query_config.yaml`.

Place Vizgen CSV pairs under `data.vizgen_data_dir` (default: `expresso_data/vizgen_cache/`). Any downloaded
`S1R1` … `S3R3` replicates are auto-discovered when `vizgen_samples` is null.

Allen MERFISH loads from a prior `aggregated_merfish.parquet` in `exploration/` when available;
otherwise re-runs with a marked warning. Allen imputed genes use `*` in combined heatmaps and
hollow markers in cross-ref scatters.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
import pandas as pd

from src.config import (
    discover_vizgen_samples,
    get_vizgen_samples,
    load_config,
    resolve_output_dir,
    restrict_config_to_genes,
    start_run,
)
from src.data_loaders import (
    get_abc_cache,
    check_vizgen_genes,
    load_vizgen_aggregated,
    load_allen_merfish_aggregate,
    merfish_gene_source_map,
    merge_crossref_aggregates,
    family_gene_region_matrix_merfish,
    combined_heatmap_matrix,
)
from src.plotting import (
    plot_family_heatmap,
    plot_combined_heatmap,
    plot_crossref_family_scatters,
    plot_crossref_side_by_side_heatmaps,
    IMPUTED_GENE_MARKER,
)
from src.utils import print_path

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "query_config.yaml"
config = load_config(CONFIG_PATH)
EXPLORATION_ROOT = resolve_output_dir(cfg=config)
config["_dataset_modality"] = "vizgen"

vizgen_samples = get_vizgen_samples(config)
config["_vizgen_samples_used"] = vizgen_samples

OUTPUT_DIR = start_run(
    PROJECT_ROOT,
    config,
    dataset="Vizgen-MERFISH",
    exploration_root=EXPLORATION_ROOT,
    notebook="03_vizgen_crossref",
)
assert not str(OUTPUT_DIR).startswith(str(PROJECT_ROOT)), (
    f"OUTPUT_DIR must be outside the repo; got {OUTPUT_DIR}"
)

print(f"Genes: {len(config['_all_genes'])}")
print(f"Brain areas: {config['brain_areas']}")
print(f"Cell type level: {config['cell_type_level']}")
filt = config.get("cell_type_name_filter") or []
print(f"Cell type name filter: {filt if filt else '(none — all types)'}")
print(f"Vizgen samples: {vizgen_samples}")
print_path("Run dir:", OUTPUT_DIR)
print_path("Manifest:", OUTPUT_DIR / "run_manifest.json")


In [ ]:
cache = get_abc_cache(config)
print("Manifest:", cache.current_manifest)

available = discover_vizgen_samples(config)
pending = sorted(set(f"S{s}R{r}" for s in range(1, 4) for r in range(1, 4)) - set(available))
print(f"\nVizgen cache: {len(available)} / 9 sample pairs present")
print(f"  Using: {vizgen_samples}")
if pending:
    print(f"  Not yet downloaded (skipped): {pending}")

In [ ]:
requested = list(config["_all_genes"])
genes_flat_orig = dict(config["_genes_flat"])
panel_ref = vizgen_samples[0]
availability = check_vizgen_genes(config, requested, panel_ref)

print(f"In Vizgen panel ({panel_ref}, n≈483): {len(availability['present'])}")
if availability["missing"]:
    print(f"Missing ({len(availability['missing'])}):")
    for gene in availability["missing"]:
        print(f"  {gene} ({genes_flat_orig.get(gene, 'unknown')})")

if not availability["present"]:
    raise RuntimeError("No requested genes in Vizgen panel.")

agg_long = load_vizgen_aggregated(cache, config)
loaded_genes = sorted(agg_long["gene"].unique())
restrict_config_to_genes(config, loaded_genes)

print(f"\nProceeding with {len(config['_all_genes'])} / {len(requested)} genes.")
print(f"Families with data: {config['_families']}")
print(f"Aggregated rows (mean across samples): {len(agg_long):,}")
print(agg_long.head())

In [ ]:
print_path("Saving Vizgen heatmaps to", OUTPUT_DIR)

for family in config["_families"]:
    mat = family_gene_region_matrix_merfish(agg_long, family, config)
    if mat.empty:
        warnings.warn(f"No data for family {family!r}; skipping heatmap.")
        continue
    path = plot_family_heatmap(family, mat, config, output_dir=OUTPUT_DIR)
    print_path("Saved", path)


In [ ]:
combined = combined_heatmap_matrix(agg_long, config)
print(f"Combined heatmap: {combined.shape[0]} cell types × {combined.shape[1]} genes")
path = plot_combined_heatmap(combined, config, output_dir=OUTPUT_DIR)
print_path("Saved", path)


In [ ]:
if config["output"].get("save_processed_data", True):
    parquet_path = OUTPUT_DIR / "aggregated_vizgen.parquet"
    agg_long.to_parquet(parquet_path, index=False)
    print_path("Saved aggregated Vizgen matrix to", parquet_path)


## Cross-reference: Allen MERFISH vs Vizgen

Loads Allen MERFISH aggregates from the newest matching run folder under `exploration/`
(pattern: `{timestamp}_{cell_type_level}_{merfish_dataset}/aggregated_merfish.parquet`).
Falls back to re-running Allen aggregation with a marked warning if not found.

In [ ]:
allen_agg, allen_parquet_path, allen_reran = load_allen_merfish_aggregate(
    cache,
    config,
    exploration_root=EXPLORATION_ROOT,
)

if allen_parquet_path is not None:
    print_path("Loaded Allen MERFISH aggregates from:", allen_parquet_path)
else:
    print("Allen MERFISH aggregates computed in this session (no prior parquet found).")

overlap_genes = sorted(set(allen_agg["gene"]) & set(agg_long["gene"]))
print(f"\nOverlapping genes for cross-ref: {len(overlap_genes)} / {len(config['_all_genes'])}")

allen_sources = merfish_gene_source_map(cache, overlap_genes, config)
config["_allen_gene_sources"] = allen_sources
n_imputed = sum(1 for s in allen_sources.values() if s == "imputed")
if n_imputed:
    imputed_genes = [g for g, s in allen_sources.items() if s == "imputed"]
    print(f"Allen imputed genes in overlap ({n_imputed}): "
          f"{[g + IMPUTED_GENE_MARKER for g in imputed_genes[:12]]}"
          f"{'...' if n_imputed > 12 else ''}")

crossref = merge_crossref_aggregates(
    allen_agg, agg_long, allen_sources, other_key="vizgen",
)
print(f"Cross-reference rows (cell_type × brain_area × gene): {len(crossref):,}")
if crossref.empty:
    raise RuntimeError("No overlapping Allen/Vizgen expression rows for cross-reference.")
print(crossref.head())


In [ ]:
heatmap_paths = plot_crossref_side_by_side_heatmaps(
    allen_agg,
    agg_long,
    config,
    output_dir=OUTPUT_DIR,
)
for path in heatmap_paths:
    print_path("Saved", path)


In [ ]:
scatter_paths = plot_crossref_family_scatters(
    crossref,
    config,
    output_dir=OUTPUT_DIR,
    file_prefix="crossref_allen_vizgen",
    other_expression_col="vizgen_expression",
    other_short_name="Vizgen",
)
for path in scatter_paths:
    print_path("Saved", path)


In [ ]:
if config["output"].get("save_processed_data", True):
    crossref_path = OUTPUT_DIR / "crossref_allen_vizgen.parquet"
    crossref.to_parquet(crossref_path, index=False)
    print_path("Saved cross-reference table to", crossref_path)


---
## Dev / smoke test (2 genes × 2 regions × S1R1 only)

Run this cell only to validate the pipeline with a smaller footprint.

In [ ]:
# test_config = load_config(CONFIG_PATH)
# test_config["brain_areas"] = ["STR", "TH"]
# test_config["receptors"] = {"dopamine": ["Drd1", "Drd2"]}
# genes_map = {}
# for fam, glist in test_config["receptors"].items():
#     for g in glist:
#         genes_map[g] = fam
# test_config["_genes_flat"] = genes_map
# test_config["_all_genes"] = list(genes_map)
# test_config["_families"] = list(test_config["receptors"].keys())
# test_config["data"]["vizgen_samples"] = ["S1R1"]
# test_config["_dataset_modality"] = "vizgen"
# test_config["_vizgen_samples_used"] = test_config["data"]["vizgen_samples"]
#
# test_cache = get_abc_cache(test_config)
# test_agg = load_vizgen_aggregated(test_cache, test_config)
# test_mat = family_gene_region_matrix_merfish(test_agg, "dopamine", test_config)
# plot_family_heatmap("dopamine", test_mat, test_config, output_dir=OUTPUT_DIR)
# print(f"Test heatmap saved to {OUTPUT_DIR / 'heatmap_dopamine.png'}")